# 06: Bayesian Analysis (Learning Outcome 6)

**Learning Outcome 6 (LO6):** Bayesian inference and posterior estimation

**Objective:** Estimate posterior distribution of monitoring effect on farm resilience

**Model:** Beta-Binomial conjugate prior for P(monitoring benefit | data)

In [ ]:
# Load data
df = pd.read_csv('data/processed/final_dataset.csv')

# Extract monitoring groups
regular = df[df['regular_monitoring'] == 1].dropna(subset=['effective_response'])
irregular = df[df['regular_monitoring'] == 0].dropna(subset=['effective_response'])

print(f"Regular monitoring observations: {len(regular)}")
print(f"Irregular monitoring observations: {len(irregular)}")

# Count successes (effective responses)
successes_regular = (regular['effective_response'] == 1).sum()
total_regular = len(regular)

successes_irregular = (irregular['effective_response'] == 1).sum()
total_irregular = len(irregular)

print(f"\nEffective responses:")
print(f"  Regular: {successes_regular}/{total_regular}")
print(f"  Irregular: {successes_irregular}/{total_irregular}")

In [ ]:
# Bayesian Analysis: Irregular Monitoring Group
print("\n" + "="*70)
print("BAYESIAN INFERENCE: P(Effective Response | Irregular Monitoring)")
print("="*70)

# Posterior for irregular group
alpha_post_irreg = alpha_prior + successes_irregular
beta_post_irreg = beta_prior + (total_irregular - successes_irregular)

posterior_mean_irreg = alpha_post_irreg / (alpha_post_irreg + beta_post_irreg)
posterior_sd_irreg = np.sqrt((alpha_post_irreg * beta_post_irreg) / 
                             ((alpha_post_irreg + beta_post_irreg)**2 * 
                              (alpha_post_irreg + beta_post_irreg + 1)))

ci_low_irreg = stats.beta.ppf(0.025, alpha_post_irreg, beta_post_irreg)
ci_high_irreg = stats.beta.ppf(0.975, alpha_post_irreg, beta_post_irreg)

print(f"Posterior: Beta({alpha_post_irreg}, {beta_post_irreg})")
print(f"\nPosterior Statistics:")
print(f"  Mean: {posterior_mean_irreg:.4f}")
print(f"  SD: {posterior_sd_irreg:.4f}")
print(f"  95% Credible Interval: [{ci_low_irreg:.4f}, {ci_high_irreg:.4f}]")

print(f"\n=== Comparison ===")
print(f"Monitoring Effect: {posterior_mean - posterior_mean_irreg:.4f}")
print(f"Regular monitors are {((posterior_mean / (posterior_mean_irreg + 1e-10)) - 1) * 100:.1f}% more likely to respond effectively")

In [ ]:
# Posterior simulation
np.random.seed(42)
samples_regular = np.random.beta(alpha_post, beta_post, 10000)
samples_irregular = np.random.beta(alpha_post_irreg, beta_post_irreg, 10000)

# P(Regular > Irregular)
prob_regular_better = (samples_regular > samples_irregular).mean()

print(f"\n=== Posterior Simulation (10,000 samples) ===")
print(f"P(Regular Monitoring > Irregular Monitoring) = {prob_regular_better:.4f}")
print(f"\nInterpretation: There is a {prob_regular_better*100:.1f}% probability that regular")
print(f"monitoring leads to higher effectiveness compared to irregular monitoring.")

In [ ]:
# Key insight
print("\n" + "="*70)
print("KEY FINDINGS: Learning Outcome 6")
print("="*70)
print(f"""
1. POSTERIOR DISTRIBUTION:
   - Regular monitoring: P(effective) ~ Beta({alpha_post}, {beta_post})
   - Mean effectiveness: {posterior_mean:.2%}
   - 95% Credible Interval: [{ci_low:.2%}, {ci_high:.2%}]

2. COMPARISON:
   - Irregular monitoring mean: {posterior_mean_irreg:.2%}
   - Monitoring benefit: {(posterior_mean - posterior_mean_irreg):.2%} percentage points

3. PROBABILITY OF SUPERIORITY:
   - P(Regular > Irregular): {prob_regular_better:.2%}
   - Strong evidence for monitoring effect

4. INTERPRETATION:
   Regular farm monitoring is associated with a {(posterior_mean / (posterior_mean_irreg + 1e-10) - 1)*100:.1f}% 
   higher probability of effective response to challenges.
""")